# Austin Bikeshare station demand-surge scoring

This jump-start project adapts the entity-by-day, 90-day decomposition-linear workflow from `BreakoutLeadGenerator_v2.ipynb` to a different domain: predicting whether a bike-share station's next-day departures will surge above its recent baseline.

**Selected public source:** Austin Bikeshare trips (`bigquery-public-data.austin_bikeshare.bikeshare_trips`). The notebook assumes a copy already exists in your GCP project and reads the copied table named by `BQ_SOURCE_TABLE`.

Why this is a close methodological match: seller → station; daily sales/refunds → departures/arrivals/duration/member mix; breakout lead → station replenishment lead. The public source has entity identifiers, timestamps, counts, durations, and customer types, so it supports multichannel daily sequences and an operational binary score.

Other viable candidates:

1. **NOAA GSOD weather** (`bigquery-public-data.noaa_gsod.gsod*`): station-day temperature, precipitation, wind, and pressure; predict extreme-weather days. Excellent multivariate history, but yearly sharded tables add SQL complexity.
2. **NYC Citi Bike** (`bigquery-public-data.new_york.citibike_trips`): station-level demand surges with a larger network. Very close fit, though query volume can be higher.
3. **Chicago taxi trips** (`bigquery-public-data.chicago_taxi_trips.taxi_trips`): community-area demand spikes using trip counts, fares, tips, and durations. Rich features, but geography/null handling needs more care.
4. **TheLook e-commerce** (`bigquery-public-data.thelook_ecommerce.order_items`): product/category demand breakouts. Closest business analogy, but the dataset is synthetic rather than observational.

Run the configuration and authentication cells first. Querying and loading data can incur GCP charges.

In [ ]:
# Install separately if needed:
# %pip install google-cloud-bigquery google-cloud-storage db-dtypes pandas numpy scikit-learn torch

import json
import os
import subprocess
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
from numpy import array, dstack
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler

from google.cloud.bigquery.client import Client
from google.cloud import bigquery, storage
from google.oauth2 import service_account

pd.set_option("display.max_columns", 50)
torch.manual_seed(42)
np.random.seed(42)

## Configuration

Set these environment variables before running:

- `GCP_SRV_SEED`: passphrase used when `srv.enc` was created.
- `GCP_PROJECT_ID`: project used for query billing and destination data.
- `BQ_SOURCE_TABLE`: copied source table, e.g. `my-project.mobility.austin_bikeshare_trips`.
- `BQ_OUTPUT_TABLE`: destination score table, e.g. `my-project.mobility.station_demand_scores`.
- `GCS_BUCKET` and `GCS_PREFIX`: existing bucket and subfolder.
- Optional `GCP_CREDENTIAL_DIR`: directory containing `srv.enc` (defaults to the home directory).

The OpenSSL options below must match those used to create `srv.enc`: AES-256-CBC, base64 armor, PBKDF2. The passphrase is passed through the child process environment, not interpolated into a shell command.

In [ ]:
project_id = os.environ["GCP_PROJECT_ID"]
source_table = os.environ["BQ_SOURCE_TABLE"]
output_table = os.environ["BQ_OUTPUT_TABLE"]
bucket_name = os.environ["GCS_BUCKET"]
gcs_prefix = os.environ.get("GCS_PREFIX", "austin_bikeshare_demand").strip("/")

credential_dir = Path(os.environ.get("GCP_CREDENTIAL_DIR", os.environ["HOME"]))
encrypted_key = credential_dir / "srv.enc"
plain_key = credential_dir / "srv.plain"
model_path = Path("austin_bikeshare_demand_model.pt").resolve()

study_end = pd.Timestamp(os.environ.get("STUDY_END_DATE", "2023-12-31"))
history_days = int(os.environ.get("HISTORY_DAYS", "730"))
seq_length = 90
pred_length = 1
batch_size = 128
score_threshold = 0.50

assert encrypted_key.is_file(), f"Missing encrypted service key: {encrypted_key}"
assert all(part not in source_table for part in ("`", ";", " ")), "Unsafe BQ_SOURCE_TABLE"
assert all(part not in output_table for part in ("`", ";", " ")), 'Unsafe BQ_OUTPUT_TABLE'

In [ ]:
# Decrypt only long enough to construct explicit credential and client objects.
# Client construction loads the JSON key; the clients retain credential objects after the file is removed.
decrypt_env = os.environ.copy()
try:
    subprocess.run(
        [
            "openssl", "enc", "-d", "-aes-256-cbc", "-a", "-pbkdf2",
            "-in", str(encrypted_key), "-out", str(plain_key),
            "-pass", "env:GCP_SRV_SEED",
        ],
        check=True,
        env=decrypt_env,
        capture_output=True,
        text=True,
    )
    os.chmod(plain_key, 0o600)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(plain_key)
    credentials = service_account.Credentials.from_service_account_file(str(plain_key))
    bqclient: Client = bigquery.Client(project=project_id, credentials=credentials)
    gcsclient = storage.Client(project=project_id, credentials=credentials)
finally:
    if plain_key.exists():
        plain_key.unlink()

print("Authenticated BigQuery and GCS clients; removed srv.plain.")

## Query station-day history

The query reads the table copied into your project. It uses stable fields from the Austin public schema and builds six daily model channels. Limiting to the top stations and a bounded date range controls scan and training size.

In [ ]:
query_start = (study_end - pd.Timedelta(days=history_days)).date()
query_end = study_end.date()

bq_query = f"""
WITH base AS (
  SELECT
    DATE(start_time) AS dt,
    CAST(start_station_id AS STRING) AS start_station_id,
    start_station_name,
    CAST(end_station_id AS STRING) AS end_station_id,
    duration_minutes,
    subscriber_type
  FROM `{source_table}`
  WHERE DATE(start_time) BETWEEN @query_start AND @query_end
    AND duration_minutes BETWEEN 1 AND 180
    AND start_station_id IS NOT NULL
),
top_stations AS (
  SELECT start_station_id, ANY_VALUE(start_station_name) AS station_name
  FROM base
  GROUP BY start_station_id
  HAVING COUNT(*) >= 100
  ORDER BY COUNT(*) DESC
  LIMIT 100
),
dates AS (
  SELECT dt FROM UNNEST(GENERATE_DATE_ARRAY(@query_start, @query_end)) AS dt
),
departures AS (
  SELECT
    dt, start_station_id,
    COUNT(*) AS departures,
    COUNT(DISTINCT end_station_id) AS unique_destinations,
    AVG(duration_minutes) AS avg_duration_minutes,
    SUM(duration_minutes) AS total_duration_minutes,
    COUNTIF(REGEXP_CONTAINS(LOWER(COALESCE(subscriber_type, '')), r'member|annual|local')) AS member_departures
  FROM base
  GROUP BY dt, start_station_id
),
arrivals AS (
  SELECT dt, end_station_id AS station_id, COUNT(*) AS arrivals
  FROM base
  WHERE end_station_id IS NOT NULL
  GROUP BY dt, station_id
)
SELECT
  d.dt,
  s.start_station_id AS station_id,
  s.station_name,
  COALESCE(x.departures, 0) AS departures,
  COALESCE(a.arrivals, 0) AS arrivals,
  COALESCE(x.unique_destinations, 0) AS unique_destinations,
  COALESCE(x.avg_duration_minutes, 0.0) AS avg_duration_minutes,
  COALESCE(x.total_duration_minutes, 0.0) AS total_duration_minutes,
  SAFE_DIVIDE(COALESCE(x.member_departures, 0), NULLIF(x.departures, 0)) AS member_share
FROM dates d
CROSS JOIN top_stations s
LEFT JOIN departures x ON x.dt = d.dt AND x.start_station_id = s.start_station_id
LEFT JOIN arrivals a ON a.dt = d.dt AND a.station_id = s.start_station_id
ORDER BY station_id, dt
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("query_start", "DATE", query_start),
        bigquery.ScalarQueryParameter("query_end", "DATE", query_end),
    ]
)

# Dry run first so cost is visible before execution.
dry_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_config.query_parameters = job_config.query_parameters
dry_job = bqclient.query(bq_query, job_config=dry_config)
print(f"Estimated bytes processed: {dry_job.total_bytes_processed:,}")

daily_df = bqclient.query(bq_query, job_config=job_config).result().to_dataframe()
daily_df["dt"] = pd.to_datetime(daily_df["dt"])
daily_df["member_share"] = daily_df["member_share"].fillna(0.0)
print(daily_df.shape)
daily_df.head()

In [ ]:
features = [
    "departures", "arrivals", "unique_destinations",
    "avg_duration_minutes", "total_duration_minutes", "member_share",
]

def aggregate_df(rdf):
    """One row per station, with ordered time-series lists (mirrors the original pattern)."""
    ordered = rdf.sort_values(["station_id", "dt"])
    return ordered.groupby(["station_id", "station_name"], as_index=False).agg({
        "dt": lambda x: list(x),
        "departures": lambda x: list(x),
        "arrivals": lambda x: list(x),
        "unique_destinations": lambda x: list(x),
        "avg_duration_minutes": lambda x: list(x),
        "total_duration_minutes": lambda x: list(x),
        "member_share": lambda x: list(x),
    })

agg_df = aggregate_df(daily_df)
print(agg_df.shape)
agg_df.head(2)

In [ ]:
class moving_avg(nn.Module):
    """Moving average block that highlights time-series trend."""
    def __init__(self, kernel_size, stride, pad=True):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)
        self.pad = pad

    def forward(self, x):
        if self.pad:
            front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
            end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
            x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        return x.permute(0, 2, 1)


class series_decomp(nn.Module):
    def __init__(self, kernel_size, stride):
        super().__init__()
        self.moving_avg = moving_avg(kernel_size, stride=stride)

    def forward(self, x):
        moving_mean = self.moving_avg(x)
        return x - moving_mean, moving_mean


class Model(nn.Module):
    """Decomposition-linear binary classifier. Input: [batch, sequence, channels]."""
    def __init__(self, configs):
        super().__init__()
        self.seq_len = configs["seq_len"]
        self.pred_len = configs["pred_len"]
        self.decomposition = series_decomp(kernel_size=3, stride=1)
        self.individual = configs["individual"]
        self.channels = configs["enc_in"]

        if self.individual:
            self.Linear_Seasonal = nn.ModuleList([
                nn.Linear(self.seq_len, self.pred_len) for _ in range(self.channels)
            ])
            self.Linear_Trend = nn.ModuleList([
                nn.Linear(self.seq_len, self.pred_len) for _ in range(self.channels)
            ])
        else:
            self.Linear_Seasonal = nn.Linear(self.seq_len, self.pred_len)
            self.Linear_Trend = nn.Linear(self.seq_len, self.pred_len)

        self.activation = nn.ReLU()
        self.final_layer = nn.Linear(self.channels, 1)
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        seasonal_init, trend_init = self.decomposition(x)
        seasonal_init = seasonal_init.permute(0, 2, 1)
        trend_init = trend_init.permute(0, 2, 1)

        if self.individual:
            seasonal_output = torch.zeros(
                [seasonal_init.size(0), seasonal_init.size(1), self.pred_len],
                dtype=seasonal_init.dtype, device=seasonal_init.device,
            )
            trend_output = torch.zeros(
                [trend_init.size(0), trend_init.size(1), self.pred_len],
                dtype=trend_init.dtype, device=trend_init.device,
            )
            for channel in range(self.channels):
                seasonal_output[:, channel, :] = self.Linear_Seasonal[channel](seasonal_init[:, channel, :])
                trend_output[:, channel, :] = self.Linear_Trend[channel](trend_init[:, channel, :])
        else:
            seasonal_output = self.Linear_Seasonal(seasonal_init)
            trend_output = self.Linear_Trend(trend_init)

        combined = self.activation(seasonal_output + trend_output)
        logits = self.final_layer(combined.permute(0, 2, 1)).squeeze(-1)
        return self.final_activation(logits).view(-1)

In [ ]:
class Time_series_dataset(Dataset):
    """Create rolling station windows and next-day surge labels."""
    def __init__(self, data, seq_len=90, feats=None, scale=True, inference_only=False):
        if data.empty:
            raise ValueError("Dataframe is empty")
        self.seq_len = seq_len
        self.feats = feats or ["departures"]
        self.scale = scale
        self.inference_only = inference_only
        self.samples = []
        self.metadata = []
        self._prepare_tensor_data_(data)

    def _scale_features(self, matrix):
        # Fit within each historical window: no future leakage into that sample.
        columns = []
        for channel in range(matrix.shape[1]):
            values = array(matrix[:, channel], dtype=np.float32).reshape(-1, 1)
            columns.append(StandardScaler().fit_transform(values).ravel())
        return dstack(columns).squeeze(0).astype(np.float32)

    def _prepare_tensor_data_(self, data):
        for _, row in data.reset_index(drop=True).iterrows():
            dates = list(row["dt"])
            raw = np.column_stack([array(row[f], dtype=np.float32) for f in self.feats])
            if self.inference_only:
                endpoints = [len(raw) - 1]
            else:
                endpoints = range(self.seq_len - 1, len(raw) - 1)

            for end in endpoints:
                start = end - self.seq_len + 1
                if start < 0:
                    continue
                window = raw[start:end + 1]
                x = self._scale_features(window) if self.scale else window.astype(np.float32)
                baseline = window[:, 0]
                surge_cutoff = float(baseline.mean() + baseline.std())
                label = np.float32(raw[end + 1, 0] > surge_cutoff) if end + 1 < len(raw) else np.float32(np.nan)
                self.samples.append((x, label))
                self.metadata.append({
                    "station_id": str(row["station_id"]),
                    "station_name": row["station_name"],
                    "report_dt": pd.Timestamp(dates[end]).date(),
                    "target_dt": (pd.Timestamp(dates[end]) + pd.Timedelta(days=1)).date(),
                })

    def __getitem__(self, index):
        x, y = self.samples[index]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.samples)


full_dataset = Time_series_dataset(agg_df, seq_len=seq_length, feats=features, scale=True)
sample_dates = np.array([np.datetime64(m["target_dt"], "D") for m in full_dataset.metadata])
date_ordinals = sample_dates.astype(np.int64)
cutoff_ordinal = int(np.quantile(date_ordinals, 0.8))
cutoff = np.datetime64(cutoff_ordinal, "D")
train_indices = np.flatnonzero(sample_dates < cutoff)
valid_indices = np.flatnonzero(sample_dates >= cutoff)

train_loader = DataLoader(Subset(full_dataset, train_indices), batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(Subset(full_dataset, valid_indices), batch_size=batch_size, shuffle=False)
print(f"Samples: {len(full_dataset):,}; train={len(train_indices):,}; validation={len(valid_indices):,}; cutoff={cutoff}")

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net = Model({
    "seq_len": seq_length,
    "pred_len": pred_length,
    "individual": True,
    "enc_in": len(features),
}).to(device)

train_model = os.environ.get("TRAIN_MODEL", "1") == "1"
epochs = int(os.environ.get("EPOCHS", "10"))

if train_model:
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = nn.BCELoss()
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        for x_feats, labels in train_loader:
            x_feats, labels = x_feats.to(device), labels.to(device)
            optimizer.zero_grad()
            probabilities = net(x_feats.float())
            loss = loss_fn(probabilities, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(labels)

        net.eval()
        correct = total = 0
        with torch.no_grad():
            for x_feats, labels in valid_loader:
                probabilities = net(x_feats.float().to(device)).cpu()
                correct += ((probabilities >= 0.5) == labels.bool()).sum().item()
                total += len(labels)
        print(f"epoch={epoch + 1:02d} loss={running_loss / len(train_indices):.4f} val_accuracy={correct / max(total, 1):.3f}")
    torch.save(net.state_dict(), model_path)
else:
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    net.load_state_dict(state_dict)

print(f"device={device}; weights={model_path}")

In [ ]:
# Score the latest complete 90-day history for each station.
inference_dataset = Time_series_dataset(
    agg_df, seq_len=seq_length, feats=features, scale=True, inference_only=True
)
inference_loader = DataLoader(inference_dataset, shuffle=False, batch_size=batch_size, pin_memory=True)

net.eval()
probability_batches = []
with torch.no_grad():
    for x_feats, _ in inference_loader:
        probability_batches.append(net(x_feats.float().to(device)).cpu())

probabilities = torch.cat(probability_batches).numpy()
scores_df = pd.DataFrame(inference_dataset.metadata)
scores_df["model_score"] = probabilities
scores_df["is_demand_surge_lead"] = scores_df["model_score"] >= score_threshold
scores_df["scored_at"] = datetime.now(timezone.utc)
scores_df = scores_df.sort_values("model_score", ascending=False).reset_index(drop=True)
scores_df.head(20)

## Publish through GCS and append to BigQuery

NDJSON avoids CSV type/quoting ambiguity. The load job allows new/relaxed fields, appends to the configured table, and leaves the staged object in GCS for auditability. An optional cleanup cell follows.

In [ ]:
export_path = Path("austin_bikeshare_demand_scores.ndjson").resolve()
publish_df = scores_df.copy()
publish_df["report_dt"] = publish_df["report_dt"].astype(str)
publish_df["target_dt"] = publish_df["target_dt"].astype(str)
publish_df["scored_at"] = publish_df["scored_at"].map(lambda value: value.isoformat())
publish_df.to_json(export_path, orient="records", lines=True)

object_name = f"{gcs_prefix}/{publish_df['report_dt'].max()}/station_demand_scores.ndjson"
bucket_obj = gcsclient.bucket(bucket_name)
blob = bucket_obj.blob(object_name)
blob.upload_from_filename(str(export_path), content_type="application/x-ndjson")
gcs_uri = f"gs://{bucket_name}/{object_name}"
print(gcs_uri)

schm = [
    bigquery.SchemaField("station_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station_name", "STRING"),
    bigquery.SchemaField("report_dt", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("target_dt", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("model_score", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("is_demand_surge_lead", "BOOL", mode="REQUIRED"),
    bigquery.SchemaField("scored_at", "TIMESTAMP", mode="REQUIRED"),
]
load_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    schema_update_options=[
        bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION,
        bigquery.SchemaUpdateOption.ALLOW_FIELD_RELAXATION,
    ],
    schema=schm,
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    ignore_unknown_values=True,
)
job = bqclient.load_table_from_uri(gcs_uri, output_table, job_config=load_config)
job.result()
print(f"Loaded {len(publish_df):,} score rows into {output_table}")

In [ ]:
# Optional cleanup of every staged object under the configured prefix.
# This is intentionally opt-in because deletion is destructive.
cleanup_gcs_prefix = os.environ.get("CLEANUP_GCS_PREFIX", "0") == "1"
if cleanup_gcs_prefix:
    bucket_obj = gcsclient.get_bucket(bucket_name)
    blobs_to_remove = list(gcsclient.list_blobs(bucket_obj, prefix=f"{gcs_prefix}/"))
    for staged_blob in blobs_to_remove:
        staged_blob.delete()
    print(f"Deleted {len(blobs_to_remove)} staged objects from gs://{bucket_name}/{gcs_prefix}/")
else:
    print("GCS cleanup skipped; set CLEANUP_GCS_PREFIX=1 to enable it.")

## Suggested next experiments

- Compare kernel sizes 3, 7, and 14 to test short versus weekly trend extraction.
- Replace the fixed one-standard-deviation label with a station-specific percentile.
- Add weather/event features, while fitting scalers only on historical windows.
- Compare individual channel linear layers against shared layers.
- Evaluate PR-AUC and precision at a fixed daily lead budget; accuracy alone is weak for rare surges.
- Add a calibrated threshold chosen from validation data before operational use.